# Evaluating AI Systems: Document Summarizer Demo

This notebook demonstrates a 4-layer evaluation pipeline for an LLM-powered news summarizer.
Each layer catches a different class of failure, and cheaper checks always run before expensive ones.

**Evaluation layers (in order):**
1. Deterministic checks -- structural rules that always return the same result
2. Heuristics -- rule-based signals that do not need a reference answer
3. Golden dataset -- semantic comparison against human-written reference summaries
4. LLM-as-judge -- Claude scores faithfulness, coherence, and relevance

**Pipeline behaviour:** Short-circuit on first failure. The LLM judge is only called if all prior layers pass.

## Setup

In [ ]:
# Uncomment and run once to install all dependencies
# !pip install -r requirements.txt
# !python -m spacy download en_core_web_sm

In [ ]:
import os
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import google.generativeai as genai
import anthropic
from dotenv import load_dotenv

# Load API keys from the project .env file.
dotenv_candidates = [
    Path.cwd() / ".env",
    Path("/Users/rheaajitjohn/SideProjects/media-summarizer/.env"),
]
for dotenv_path in dotenv_candidates:
    if dotenv_path.exists():
        load_dotenv(dotenv_path=dotenv_path, override=True)
        break

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
ANTHROPIC_API_KEY = (
    os.getenv("ANTHROPIC_API_KEY")
    or os.getenv("CLAUDE_API_KEY")
    or os.getenv("ANTHROPIC_KEY")
)

if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)

if ANTHROPIC_API_KEY:
    anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
else:
    anthropic_client = None

# Load models once at startup to avoid reloading on each eval call.
nlp = spacy.load("en_core_web_sm")
embedder = SentenceTransformer("all-MiniLM-L6-v2")


## Core: Shared Interface and Pipeline Runner

Every evaluator in this demo shares the same signature: `(article, summary) -> EvalResult`.
This means the pipeline runner does not need to know anything about individual evaluators --
adding a new layer is as simple as appending a function to the list.

In [ ]:
@dataclass
class EvalResult:
    """Result returned by every evaluation layer."""
    passed: bool
    reason: str
    score: float | None = None


def run_pipeline(
    evaluators: list[Callable[[str, str], EvalResult]],
    article: str,
    summary: str,
) -> EvalResult:
    """Run evaluators in order and short-circuit on the first failure."""
    for evaluator in evaluators:
        result = evaluator(article, summary)
        print(f"  [{evaluator.__name__}] passed={result.passed} | {result.reason}")
        if not result.passed:
            return result
    return EvalResult(passed=True, reason="All evaluation layers passed.")

## Sample Data

Two CNN/DailyMail-style article and reference summary pairs. Reference summaries were written by hand to serve
as ground truth for the golden dataset layer.

In [ ]:
SAMPLE_ARTICLES: list[dict[str, str]] = [
    {
        "article": (
            "WASHINGTON (CNN) -- President Barack Obama signed the Affordable Care Act into law "
            "on Tuesday, March 23, 2010, describing it as a historic moment for the United States. "
            "The legislation passed the Senate with 60 votes and the House with 219 votes, and is "
            "expected to extend health insurance coverage to an estimated 32 million Americans who "
            "are currently uninsured. Republicans in Congress unanimously opposed the bill, with "
            "House Minority Leader John Boehner calling it a step in the wrong direction for America. "
            "The law requires most Americans to purchase health insurance by 2014 or face a financial "
            "penalty. Under the new rules, insurance companies will no longer be allowed to deny "
            "coverage based on pre-existing conditions. Obama signed the bill in the East Room of "
            "the White House before hundreds of lawmakers and health care advocates."
        ),
        "reference_summary": (
            "- President Obama signed the Affordable Care Act on March 23, 2010\n"
            "- The law extends health coverage to an estimated 32 million uninsured Americans\n"
            "- Republicans unanimously opposed the bill\n"
            "- Insurance companies can no longer deny coverage for pre-existing conditions"
        ),
    },
    {
        "article": (
            "SAN FRANCISCO (CNN) -- Apple CEO Steve Jobs unveiled the first iPhone on January 9, 2007, "
            "at the Macworld Conference in San Francisco. Jobs described the device as a revolutionary "
            "product that changes everything and said it was five years ahead of any other mobile phone. "
            "The iPhone combined a phone, a widescreen iPod, and an internet communication device into "
            "a single product. It featured a multi-touch screen and no physical keyboard, a radical "
            "departure from existing smartphones at the time. The device went on sale in the United "
            "States on June 29, 2007, at a starting price of $499. Analysts predicted it would reshape "
            "the mobile phone industry, and within a year Apple had sold more than six million units."
        ),
        "reference_summary": (
            "- Apple CEO Steve Jobs unveiled the first iPhone on January 9, 2007\n"
            "- Jobs called the device five years ahead of any other mobile phone\n"
            "- The iPhone combined a phone, iPod, and internet device with a multi-touch screen\n"
            "- It went on sale June 29, 2007, and sold over six million units in its first year"
        ),
    },
]

print(f"Loaded {len(SAMPLE_ARTICLES)} sample articles.")

## Gemini Summarizer

The summarizer uses Gemini to produce bullet-point summaries of news articles.
An optional `extra_instruction` parameter lets us inject bad instructions during failure demos.

In [ ]:
DEFAULT_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")


def summarize(article: str, extra_instruction: str = "") -> str:
    """Summarize a news article in bullet points using Gemini."""
    prompt = (
        "Summarize the following news article in bullet points.\n"
        "Use '-' at the start of each bullet point.\n"
        "Keep the summary between 30 and 150 words.\n"
        f"{extra_instruction}\n\n"
        f"Article:\n{article}\n\n"
        "Summary:"
    )
    try:
        model = genai.GenerativeModel(DEFAULT_MODEL)
        response = model.generate_content(prompt)
        return response.text.strip()
    except Exception as exc:
        if not article.strip():
            return ""

        fallback = (
            "- Fallback summary used because the Gemini API was unavailable.\n"
            "- The notebook can still demonstrate the evaluation layers without a live model call."
        )
        print(f"Warning: Gemini request failed for model '{DEFAULT_MODEL}'. Using fallback summary.")
        return fallback


---
## Layer 1: Deterministic Checks

These are hardcoded structural rules with zero ambiguity. The same input always produces
the same result, and no model or external service is involved. They are the cheapest
checks to run and should always be the first gate in the pipeline.

**Rules:**
- Output must not be empty
- Word count must be between 30 and 150
- Output must contain at least 2 bullet points (lines starting with `-`)

In [ ]:
WORD_COUNT_MIN = 30
WORD_COUNT_MAX = 150
MIN_BULLETS = 2


def check_deterministic(article: str, summary: str) -> EvalResult:
    """Check that the summary is non-empty, within word limits, and uses bullet points."""
    print(f"Deterministic check config: min_words={WORD_COUNT_MIN}, max_words={WORD_COUNT_MAX}, min_bullets={MIN_BULLETS}")

    if not summary.strip():
        print("Deterministic check: summary is empty")
        return EvalResult(passed=False, reason="Summary is empty.")

    word_count = len(summary.split())
    if not (WORD_COUNT_MIN <= word_count <= WORD_COUNT_MAX):
        print(f"Deterministic check: word count {word_count} is outside the allowed range")
        return EvalResult(
            passed=False,
            reason=f"Word count {word_count} is outside the allowed range ({WORD_COUNT_MIN}-{WORD_COUNT_MAX}).",
        )

    bullets = [line for line in summary.split("\n") if line.strip().startswith("-")]
    if len(bullets) < MIN_BULLETS:
        print(f"Deterministic check: found {len(bullets)} bullet(s), need at least {MIN_BULLETS}")
        return EvalResult(
            passed=False,
            reason=f"Found {len(bullets)} bullet point(s), need at least {MIN_BULLETS}.",
        )

    print("Deterministic check: all checks passed")
    return EvalResult(passed=True, reason="Passed all deterministic checks.")

### Demo: Triggering a Deterministic Failure

Passing an empty string causes Gemini to return an empty or near-empty output.
The non-empty check catches it immediately -- no heuristics or LLM judge are called.

In [ ]:
broken_summary_1 = summarize("")
result_1_fail = check_deterministic("", broken_summary_1)

print("--- Failing example ---")
print(f"Summary output : {repr(broken_summary_1[:100])}")
print(f"Passed         : {result_1_fail.passed}")

print(f"Reason         : {result_1_fail.reason}")



print("\n--- Passing example ---")
good_summary_1 = (
    "- President Barack Obama signed the Affordable Care Act into law on March 23, 2010, describing it as a historic moment for the United States.\n"
    "- The law expanded health coverage to millions of Americans and faced strong Republican opposition in Congress."
)
print(f"Summary output : {good_summary_1}")

result_1_pass = check_deterministic("", good_summary_1)
print(f"Passed         : {result_1_pass.passed}")
print(f"Reason         : {result_1_pass.reason}")

---
## Layer 2: Heuristics

Heuristics are rule-based signals that go beyond structural checks but still do not
require a reference answer. They catch outputs that look structurally valid but are
semantically hollow or repetitive.

**Rules:**
- Named entity coverage: at least one named entity from the article (person, org, location)
  must appear in the summary -- catches generic filler responses
- No repetition: no sentence should appear more than once -- catches a common LLM failure mode

In [ ]:
def check_heuristics(article: str, summary: str) -> EvalResult:
    """Compute three heuristic scores and apply pass/flag/fail thresholds."""
    # Length ratio: output length / input length (capped at 1.0)
    article_words = article.split()
    summary_words = summary.split()
    len_ratio = min(1.0, len(summary_words) / max(1, len(article_words)))

    # Keyword overlap: fraction of named entities from the article appearing in the summary
    article_doc = nlp(article)
    summary_doc = nlp(summary)
    article_entities = {ent.text.lower() for ent in article_doc.ents}
    summary_entities = {ent.text.lower() for ent in summary_doc.ents}
    if article_entities:
        overlap = len(article_entities & summary_entities) / len(article_entities)
    else:
        overlap = 0.0

    # Repetition score: unique sentences / total sentences
    sentences = [s.strip() for s in re.split(r"[.\n]", summary) if s.strip()]
    total = len(sentences)
    unique = len(set(s.lower() for s in sentences))
    repetition_score = (unique / total) if total else 1.0

    # Print the raw scores for transparency
    print(f"Heuristic scores: length_ratio={len_ratio:.2f}, keyword_overlap={overlap:.2f}, repetition={repetition_score:.2f}")

    # Thresholds: >=0.75 -> pass, 0.50-0.74 -> flag, <0.50 -> hard fail
    def _status(score: float) -> str:
        if score < 0.5:
            return "fail"
        if score < 0.75:
            return "flag"
        return "pass"

    scores = {"length": len_ratio, "overlap": overlap, "repetition": repetition_score}
    statuses = {k: _status(v) for k, v in scores.items()}

    # Hard fail if any metric fails
    for name, st in statuses.items():
        if st == "fail":
            reason = f"Hard fail: {name} score {scores[name]:.2f} < 0.50."
            return EvalResult(passed=False, reason=reason, score=min(scores.values()))

    # If any metric is flagged, return a pass but mark it as flagged so the pipeline can continue
    flagged = [n for n, s in statuses.items() if s == "flag"]
    if flagged:
        details = ', '.join([f"{n}={scores[n]:.2f}" for n in flagged])
        reason = f"Flagged heuristics: {details}"
        return EvalResult(passed=True, reason=reason, score=min(scores.values()))

    # All metrics pass
    reason = f"Passed all heuristic thresholds: length={len_ratio:.2f}, overlap={overlap:.2f}, repetition={repetition_score:.2f}"
    return EvalResult(passed=True, reason=reason, score=min(scores.values()))

### Demo: Triggering a Heuristic Failure

Telling Gemini to respond in a single paragraph -- without bullet points -- tends to
produce generic filler that drops named entities from the article.

In [ ]:
article = SAMPLE_ARTICLES[0]["article"]

broken_summary_2 = (
    "This summary is generic and repetitive. "
    "This summary is generic and repetitive."
)
result_2 = check_heuristics(article, broken_summary_2)

print("--- Failing example ---")
print("Summary output :", broken_summary_2)
print("Passed         :", result_2.passed)
print(f"Score (min)    : {result_2.score}")
print(f"Reason         : {result_2.reason}")

good_summary_2 = (
    "- President Barack Obama signed the Affordable Care Act into law on March 23, 2010.\n"
    "- Republicans in Congress strongly opposed the measure."
)
result_2_pass = check_heuristics(article, good_summary_2)

print("\n--- Passing example ---")
print("Summary output :", good_summary_2)
print("Passed         :", result_2_pass.passed)
print(f"Score (min)    : {result_2_pass.score}")
print(f"Reason         : {result_2_pass.reason}")

---
## Layer 3: Golden Dataset

The golden dataset layer compares the model's output against a human-written reference
summary using semantic similarity. Unlike exact-match metrics such as ROUGE, cosine
similarity over sentence embeddings tolerates valid paraphrasing while still penalising
summaries that have drifted from the source material.

**Method:** `all-MiniLM-L6-v2` embeddings, cosine similarity threshold of 0.6.
Articles with no reference in the dataset are skipped (pass through).

In [ ]:
COSINE_THRESHOLD = 0.6

# Build a lookup from article text to its reference summary
_reference_lookup: dict[str, str] = {
    sample["article"]: sample["reference_summary"] for sample in SAMPLE_ARTICLES
}


def check_golden_dataset(article: str, summary: str) -> EvalResult:
    """Compare summary against a reference using cosine similarity (threshold 0.6)."""
    reference = _reference_lookup.get(article)
    print(f"Golden-dataset check: threshold={COSINE_THRESHOLD}, reference_available={reference is not None}")

    if reference is None:
        print("Golden-dataset check: no reference available, skipping")
        return EvalResult(passed=True, reason="No reference available for this article -- skipping.")

    embeddings = embedder.encode([summary, reference])
    similarity = float(cosine_similarity([embeddings[0]], [embeddings[1]])[0][0])
    print(f"Golden-dataset check: similarity={similarity:.4f}")

    if similarity < COSINE_THRESHOLD:
        print("Golden-dataset check: similarity below threshold")
        return EvalResult(
            passed=False,
            reason=f"Cosine similarity {similarity:.2f} is below threshold {COSINE_THRESHOLD}.",
            score=similarity,
        )

    print("Golden-dataset check: similarity meets threshold")
    return EvalResult(
        passed=True,
        reason=f"Cosine similarity {similarity:.2f} meets threshold {COSINE_THRESHOLD}.",
        score=similarity,
    )

### Demo: Triggering a Golden Dataset Failure

Truncating the article to 2 sentences before sending it to Gemini produces a summary
that covers only a fragment of the story. The cosine similarity against the full
reference summary drops below the threshold.

In [ ]:
article = SAMPLE_ARTICLES[0]["article"]

# Use a deliberately incomplete summary to force a golden-dataset failure
broken_summary_3 = "- The law was signed in 2010."
result_3 = check_golden_dataset(article, broken_summary_3)

print("--- Failing example ---")
print(f"Summary output : {broken_summary_3}")
print(f"Passed         : {result_3.passed}")
print(f"Score          : {result_3.score}")
print(f"Reason         : {result_3.reason}")

good_summary_3 = SAMPLE_ARTICLES[0]["reference_summary"]
result_3_pass = check_golden_dataset(article, good_summary_3)

print("\n--- Passing example ---")
print(f"Summary output : {good_summary_3}")
print(f"Passed         : {result_3_pass.passed}")
print(f"Score          : {result_3_pass.score}")
print(f"Reason         : {result_3_pass.reason}")

---
## Layer 4: LLM-as-Judge

The LLM judge is the most powerful but also the most expensive layer. It is only called
after all cheaper checks have passed. Claude evaluates the summary on three dimensions:

- **Faithfulness** -- does the summary contain only information from the article? (catches hallucinations)
- **Coherence** -- is it readable and logically structured?
- **Relevance** -- does it cover the most important parts of the article?

Each dimension is scored 1 to 5. All three must score 3 or above to pass.
Using a different model (Claude) to judge a Gemini summary avoids self-evaluation bias.

In [ ]:
JUDGE_PASS_THRESHOLD = 3
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-4-6")

_JUDGE_PROMPT = """\
You are evaluating a news article summary produced by an AI model.

Score the summary on three dimensions, each from 1 to 5:
- faithfulness: Does the summary only contain information from the article? (1=hallucinations, 5=fully faithful)
- coherence: Is the summary readable and logically structured? (1=incoherent, 5=very clear)
- relevance: Does it focus on the most important parts of the article? (1=misses key points, 5=highly relevant)

Return a JSON object in exactly this format:
{{
  "faithfulness": <score>,
  "coherence": <score>,
  "relevance": <score>,
  "reasoning": "<one sentence explanation>"
}}

Article:
{article}

Summary:
{summary}
"""


def check_llm_judge(article: str, summary: str) -> EvalResult:
    """Score the summary using Claude on faithfulness, coherence, and relevance (all must be >= 3)."""
    print(f"LLM judge: model={CLAUDE_MODEL}, pass_threshold={JUDGE_PASS_THRESHOLD}")

    if anthropic_client is None:
        print("LLM judge: Anthropic client not configured")
        return EvalResult(
            passed=False,
            reason="Anthropic client is not configured because no API key was found.",
            score=None,
        )

    try:
        message = anthropic_client.messages.create(
            model=CLAUDE_MODEL,
            max_tokens=256,
            messages=[
                {
                    "role": "user",
                    "content": _JUDGE_PROMPT.format(article=article, summary=summary),
                }
            ],
        )
    except anthropic.AuthenticationError as exc:
        print(f"LLM judge: authentication failed -> {exc}")
        return EvalResult(
            passed=False,
            reason=f"Anthropic authentication failed: {exc}",
            score=None,
        )
    except Exception as exc:
        print(f"LLM judge: request failed -> {exc}")
        return EvalResult(
            passed=False,
            reason=f"Anthropic request failed: {exc}",
            score=None,
        )

    raw_text = message.content[0].text if getattr(message, "content", None) else ""
    try:
        scores = json.loads(raw_text)
    except json.JSONDecodeError:
        print("LLM judge: returned non-JSON output")
        return EvalResult(
            passed=False,
            reason=f"Claude returned non-JSON output: {raw_text[:200]}",
            score=None,
        )

    print(f"LLM judge scores: {scores}")
    dimensions = ["faithfulness", "coherence", "relevance"]
    failing = [dim for dim in dimensions if scores[dim] < JUDGE_PASS_THRESHOLD]

    if failing:
        print(f"LLM judge: failing dimensions -> {failing}")
        return EvalResult(
            passed=False,
            reason=f"Failed on: {', '.join(failing)}. Reasoning: {scores['reasoning']}",
            score=float(min(scores[dim] for dim in dimensions)),
        )

    print("LLM judge: all dimensions passed")
    return EvalResult(
        passed=True,
        reason=f"All dimensions passed. Reasoning: {scores['reasoning']}",
        score=float(min(scores[dim] for dim in dimensions)),
    )

### Demo: Triggering an LLM Judge Failure

Generate a good summary first, then inject a factually wrong sentence into it.
The output passes all structural and heuristic checks, but Claude's faithfulness
score tanks because the injected claim contradicts the article.

In [ ]:
article = SAMPLE_ARTICLES[0]["article"]
good_summary = summarize(article)

# Injecting a contradictory sentence that no structural check would catch
broken_summary_4 = good_summary + "\n- The legislation was signed in 1823."
result_4 = check_llm_judge(article, broken_summary_4)

print("--- Failing example (contradiction injected) ---")
print("Summary output:")
print(broken_summary_4)
print("")
print(f"Passed         : {result_4.passed}")
print(f"Min score      : {result_4.score}")
print(f"Reason         : {result_4.reason}")

# correct summary that should pass the judge
good_summary_4 = (
    "- President Barack Obama signed the Affordable Care Act into law on March 23, 2010.\n"
    "- The law expanded health coverage and was opposed by Republicans in Congress."
)
result_4_pass = check_llm_judge(article, good_summary_4)

print("\n--- Passing example ---")
print("Summary output:")
print(good_summary_4)
print("")
print(f"Passed         : {result_4_pass.passed}")
print(f"Min score      : {result_4_pass.score}")
print(f"Reason         : {result_4_pass.reason}")

---
## Full Pipeline: All Layers Together

The evaluators are composed into a single ordered list. `run_pipeline` iterates through
them, short-circuiting on the first failure. Each layer name and result is printed so
the audience can see exactly where the pipeline stopped.

In [ ]:
EVALUATORS: list[Callable[[str, str], EvalResult]] = [
    check_deterministic,
    check_heuristics,
    check_golden_dataset,
    check_llm_judge,
]

article = SAMPLE_ARTICLES[0]["article"]
good_summary = summarize(article)

print("--- Running pipeline on a well-formed summary ---")
final_result = run_pipeline(EVALUATORS, article, good_summary)
print(f"\nFinal result -- Passed: {final_result.passed}")
print(f"Reason       : {final_result.reason}")

In [ ]:
# Running the same pipeline on a broken summary to watch it short-circuit
print("--- Running pipeline on an empty input ---")
broken_result = run_pipeline(EVALUATORS, "", "")
print(f"\nFinal result -- Passed: {broken_result.passed}")
print(f"Reason       : {broken_result.reason}")